# L11 — Statystyka opisowa w Pythonie

**Programowanie w Pythonie II** | Laboratorium 11  
Dataset: generowany w notebooku — 200 pracowników HR

## Setup — dane HR (200 pracowników)

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
n = 200

dzialy = np.random.choice(['IT', 'Sprzedaz', 'HR', 'Marketing', 'Finanse'], n,
                          p=[0.30, 0.25, 0.15, 0.20, 0.10])
staz = np.random.gamma(shape=3, scale=2, size=n).clip(0.5, 20).round(1)
baza = {'IT': 9000, 'Sprzedaz': 7000, 'HR': 6500, 'Marketing': 7500, 'Finanse': 8500}
wynagrodzenie = np.array([
    baza[d] + staz[i] * 300 + np.random.normal(0, 1200)
    for i, d in enumerate(dzialy)
]).clip(4000, 25000).round(-2)

# 5 celowo wstawionych outlierów (błędy danych / kontrakty specjalne)
wynagrodzenie[np.random.choice(n, 5, replace=False)] = np.random.choice(
    [2000, 2500, 35000, 40000, 38000], 5, replace=False
)

df = pd.DataFrame({
    'dzial': dzialy,
    'staz_lat': staz,
    'wynagrodzenie': wynagrodzenie,
    'wiek': (25 + staz + np.random.normal(0, 3, n)).clip(22, 65).round().astype(int),
    'ocena_roczna': np.random.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.10, 0.40, 0.35, 0.10])
})

print(f"Dataset HR: {df.shape[0]} pracowników, {df.shape[1]} kolumn")
print(df.head())
print("\nTypy kolumn:")
print(df.dtypes)

## Ćwiczenie 1: Statystyki opisowe na danych biznesowych

**Kontekst biznesowy:** Jesteś analitykiem w dziale HR. Dyrektor personalny pyta: "Jak wyglądają nasze wynagrodzenia? Czy są zróżnicowane? Który dział płaci najlepiej?"

### 1a. Miary tendencji centralnej

In [ ]:
placa = df['wynagrodzenie']

srednia = placa.mean()
mediana = placa.median()
dominanta = placa.mode().iloc[0]

print("=== MIARY TENDENCJI CENTRALNEJ ===")
print(f"Średnia:   {srednia:>10,.0f} PLN")
print(f"Mediana:   {mediana:>10,.0f} PLN")
print(f"Dominanta: {dominanta:>10,.0f} PLN")

# Mediana < Średnia → rozkład prawostronnie skośny.
# Outlierzy z bardzo wysokimi pensjami (35k–40k) ciągną średnią w górę,
# mediana pozostaje odporna bo opiera się na wartości środkowej.

### 1b. Miary rozproszenia

In [ ]:
odch_std = placa.std()
q1 = placa.quantile(0.25)
q3 = placa.quantile(0.75)
iqr = q3 - q1
rozstep = placa.max() - placa.min()

print("=== MIARY ROZPROSZENIA ===")
print(f"Odchylenie std: {odch_std:>10,.0f} PLN")
print(f"Q1 (P25):       {q1:>10,.0f} PLN")
print(f"Q3 (P75):       {q3:>10,.0f} PLN")
print(f"IQR:            {iqr:>10,.0f} PLN")
print(f"Rozstęp:        {rozstep:>10,.0f} PLN")

# Rozstęp jest tak duży (ok. 38 000 PLN) bo wstawione outliery sięgają 40 000 PLN,
# a dolna granica to ~2 000 PLN. Rozstęp = max - min, więc jedna skrajna wartość
# całkowicie zniekształca tę miarę.

### 1c. Statystyki per dział

In [ ]:
dzialy_stats = df.groupby('dzial', observed=True)['wynagrodzenie'].agg([
    'mean', 'median', 'std'
]).round(0).sort_values('median', ascending=False)
dzialy_stats.columns = ['Średnia', 'Mediana', 'Std']

print("=== WYNAGRODZENIA PER DZIAŁ ===")
print(dzialy_stats)

### 1d. Wizualizacja — histogram z miarami centralnymi

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(placa, bins=30, color='steelblue', alpha=0.8, edgecolor='white')
ax.axvline(srednia,   color='red',    linestyle='--', lw=2, label=f'Średnia: {srednia:,.0f}')
ax.axvline(mediana,   color='green',  linestyle='-',  lw=2, label=f'Mediana: {mediana:,.0f}')
ax.axvline(dominanta, color='orange', linestyle=':',  lw=2, label=f'Dominanta: {dominanta:,.0f}')

ax.set_title('Rozkład wynagrodzeń — miary tendencji centralnej')
ax.set_xlabel('Wynagrodzenie (PLN)')
ax.set_ylabel('Liczba pracowników')
ax.legend()
plt.tight_layout()
plt.show()
plt.close()

## Ćwiczenie 2: Analiza korelacji

**Kontekst biznesowy:** Dyrektor finansowy pyta: "Co decyduje o wysokości wynagrodzenia — staż pracy, wiek, a może ocena roczna?"

### 2a. Korelacja Pearsona: staż vs wynagrodzenie

In [ ]:
r, p_value = stats.pearsonr(df['staz_lat'], df['wynagrodzenie'])

print(f"Korelacja Pearsona (staż–wynagrodzenie):")
print(f"  r = {r:.4f}")
print(f"  p = {p_value:.4f}")
print(f"  Interpretacja: {'istotna' if p_value < 0.05 else 'nieistotna'} statystycznie")

### 2b. Korelacja Spearmana — porównanie

In [ ]:
rho, p_rho = stats.spearmanr(df['staz_lat'], df['wynagrodzenie'])

print(f"Korelacja Spearmana (staż–wynagrodzenie):")
print(f"  rho = {rho:.4f}")
print(f"  p   = {p_rho:.4f}")
print(f"\nPorównanie: Pearson r={r:.3f} vs Spearman rho={rho:.3f}")
print(f"Różnica: {abs(rho - r):.3f} — {'duże' if abs(rho-r) > 0.1 else 'małe'} rozbieżności")

# Spearman jest wyższy niż Pearson, bo operuje na rangach i jest odporny na outliery.
# Outliery (np. 2 000 PLN lub 40 000 PLN) zaburzają liniowość i obniżają Pearsona,
# natomiast Spearman mierzy tylko monotoniczność (kolejność), więc mniej na nie reaguje.

### 2c. Macierz korelacji — wszystkie zmienne

In [ ]:
corr = df[['staz_lat', 'wynagrodzenie', 'wiek', 'ocena_roczna']].corr()
print("Macierz korelacji Pearsona:")
print(corr.round(3))

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)

for i in range(len(corr)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f'{corr.values[i, j]:.2f}',
                ha='center', va='center', fontsize=11,
                color='black' if abs(corr.values[i, j]) < 0.7 else 'white')

ax.set_title('Macierz korelacji — dataset HR')
plt.tight_layout()
plt.show()
plt.close()

### 2d. Korelacja: marketing spend vs revenue

In [ ]:
np.random.seed(100)
n_mkt = 50
marketing_spend = np.random.uniform(10000, 100000, n_mkt)
revenue = marketing_spend * 5.2 + np.random.normal(0, 30000, n_mkt)
revenue = revenue.clip(0)

r_mkt, p_mkt = stats.pearsonr(marketing_spend, revenue)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(marketing_spend, revenue, alpha=0.6, color='coral', s=50)

z = np.polyfit(marketing_spend, revenue, 1)
p_fit = np.poly1d(z)
x_line = np.linspace(marketing_spend.min(), marketing_spend.max(), 100)
ax.plot(x_line, p_fit(x_line), 'b--', lw=2, label=f'Trend (r={r_mkt:.2f})')

ax.set_title('Wydatki na marketing vs Przychody')
ax.set_xlabel('Wydatki na marketing (PLN)')
ax.set_ylabel('Przychody (PLN)')
ax.legend()
plt.tight_layout()
plt.show()
plt.close()

print(f"Marketing spend vs Revenue: r = {r_mkt:.4f}, p = {p_mkt:.2e}")

## Ćwiczenie 3: Pełna analiza statystyczna datasetu HR

**Kontekst biznesowy:** Przygotowujesz kompleksowy raport HR dla zarządu — opis rozkładu, analiza percentylowa, porównanie działów.

### 3a. scipy.stats.describe — szybki przegląd rozkładu

In [ ]:
opis = stats.describe(df['wynagrodzenie'])

print("=== scipy.stats.describe — wynagrodzenie ===")
print(f"Liczba obs.:  {opis.nobs}")
print(f"Min / Max:    {opis.minmax[0]:,.0f} / {opis.minmax[1]:,.0f} PLN")
print(f"Średnia:      {opis.mean:,.0f} PLN")
print(f"Wariancja:    {opis.variance:,.0f}")
print(f"Skośność:     {opis.skewness:.4f}")
print(f"Kurtoza:      {opis.kurtosis:.4f}")
print()
# Skośność > 1.0 → silnie prawostronny rozkład: długi prawy ogon z outlierami (35k–40k PLN).
# Kurtoza > 0 → spiczasty rozkład (leptokurtyczny): dużo obserwacji blisko środka
# i jednocześnie grube ogony — typowe gdy mamy skupiony rdzeń + kilka ekstremalnych wartości.

### 3b. Analiza percentylowa

In [ ]:
poziomy = [10, 25, 50, 75, 90, 95, 99]
percentyle = np.percentile(df['wynagrodzenie'], poziomy)

print("=== ANALIZA PERCENTYLOWA — wynagrodzenie ===")
for p_lvl, val in zip(poziomy, percentyle):
    print(f"  P{p_lvl:>2}: {val:>10,.0f} PLN")

p75_val = np.percentile(df['wynagrodzenie'], 75)
powyzej_12k = (df['wynagrodzenie'] > 12000).mean() * 100

print(f"\nP75 (benchmark): {p75_val:,.0f} PLN")
print(f"Pracownicy powyżej 12 000 PLN: {powyzej_12k:.1f}%")

### 3c. Analiza per dział — pełna tabela

In [ ]:
pelna_tabela = df.groupby('dzial', observed=True)['wynagrodzenie'].agg([
    'mean', 'median', 'std', 'min', 'max',
    ('IQR', lambda x: x.quantile(0.75) - x.quantile(0.25))
]).round(0).sort_values('median', ascending=False)

pelna_tabela.columns = ['Średnia', 'Mediana', 'Std', 'Min', 'Max', 'IQR']
print("=== PEŁNA TABELA STATYSTYK PER DZIAŁ ===")
print(pelna_tabela)

### 3d. Wizualizacja — boxplot per dział

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
df.boxplot(column='wynagrodzenie', by='dzial', ax=ax)

mediana_globalna = df['wynagrodzenie'].median()
ax.axhline(mediana_globalna, color='red', linestyle='--', lw=2, label=f'Mediana globalna: {mediana_globalna:,.0f} PLN')

ax.set_title('Rozkład wynagrodzeń per dział')
ax.set_xlabel('Dział')
ax.set_ylabel('Wynagrodzenie (PLN)')
ax.legend()
plt.suptitle('')
plt.tight_layout()
plt.show()
plt.close()

### 3e. scipy.stats.describe — per dział IT i HR

In [ ]:
for d in ['IT', 'HR']:
    podzb = df[df['dzial'] == d]['wynagrodzenie']
    opis_d = stats.describe(podzb)
    print(f"\n{d} (n={opis_d.nobs}):")
    print(f"  Skośność: {opis_d.skewness:.3f}")
    print(f"  Kurtoza:  {opis_d.kurtosis:.3f}")

# IT: silnie prawostronny (duża skośność) z powodu outlierów (35k–40k PLN)
# trafionych do IT — mocno podnoszą prawy ogon.
# HR: prawie symetryczny (skośność bliska 0) — brak ekstremalnych wynagrodzeń,
# mniejszy dział o bardziej jednorodnej strukturze płac.

## Ćwiczenie 4: Wykrywanie outlierów

**Kontekst biznesowy:** Zidentyfikuj pracowników z anomalnymi wynagrodzeniami przed finalnym raportem.

### 4a. Metoda IQR (reguła Tukeya)

In [ ]:
q1 = df['wynagrodzenie'].quantile(0.25)
q3 = df['wynagrodzenie'].quantile(0.75)
iqr = q3 - q1
dolna = q1 - 1.5 * iqr
gorna = q3 + 1.5 * iqr

print(f"IQR granice: [{dolna:,.0f} PLN, {gorna:,.0f} PLN]")

maska_iqr = (df['wynagrodzenie'] < dolna) | (df['wynagrodzenie'] > gorna)
outliery = df[maska_iqr]
print(f"Liczba outlierów: {maska_iqr.sum()} z {len(df)}")
print("\nSzczegóły outlierów:")
print(outliery[['dzial', 'staz_lat', 'wynagrodzenie']].to_string())

### 4b. Metoda z-score

In [ ]:
z_scores = np.abs(stats.zscore(df['wynagrodzenie']))
maska_z = z_scores > 3.0

outliery_z = df[maska_z]
print(f"Outlierzy z-score (|z| > 3.0): {maska_z.sum()} obserwacji")
print(outliery_z[['dzial', 'staz_lat', 'wynagrodzenie']].to_string())

### 4c. Wpływ outlierów na statystyki

In [ ]:
bez_outlierow = df[~maska_iqr]['wynagrodzenie']
z_outlierami  = df['wynagrodzenie']

print(f"{'Miara':<20} {'Z outlierami':>15} {'Bez outlierów':>15} {'Zmiana':>10}")
print("-" * 62)
for nazwa, f_z, f_bez in [
    ('Średnia',  z_outlierami.mean(),   bez_outlierow.mean()),
    ('Mediana',  z_outlierami.median(), bez_outlierow.median()),
    ('Std',      z_outlierami.std(),    bez_outlierow.std()),
]:
    zmiana = f_z - f_bez
    print(f"{nazwa:<20} {f_z:>15,.0f} {f_bez:>15,.0f} {zmiana:>+10,.0f}")

# Mediana jest najbardziej stabilna — zmienia się o kilkadziesiąt PLN.
# Std zmienia się ponad 2x — kwadrat odchylenia mocno amplifikuje ekstrema.
# Średnia jest wrażliwa, ale mniej niż std.